# Notebook 02 — Data Preprocessing

**Project:** Blood Donation Prediction System  
**Authors:** Mihret Alemayehu, Abebech Nega  
**Institution:** Debre Berhan University  
**Instructor:** Petros Abebe  

---

## Objective
Transform the raw dataset into a clean, model-ready form by:
1. Standardizing missing value representations
2. Removing exact duplicate records
3. Imputing missing values using **K-Nearest Neighbors (k=5)**
4. Encoding categorical variables (Blood Type, Location)
5. Engineering derived RFMTC features
6. Saving the processed dataset to `data/processed/`


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Add project root to Python path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.loader import DataLoader
from src.data.cleaner import DataCleaner
from src.features.builder import FeatureBuilder

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline
print('Imports successful.')

---
## 1. Load Raw Data


In [ ]:
loader = DataLoader(data_dir=os.path.join(PROJECT_ROOT, 'data', 'raw'))
df_raw = loader.load_csv('blood_donation.csv')

print(f'Raw data shape: {df_raw.shape}')
print(f'Total missing : {df_raw.isnull().sum().sum():,}')
df_raw.head()

---
## 2. Standardize Missing Value Representations


In [ ]:
# Initialize our cleaning class
# n_neighbors=5 is specified in the course guidelines for KNN imputation
cleaner = DataCleaner(n_neighbors=5)

# Replace empty strings, 'NA', 'N/A', '--' etc. with np.nan
df_standardized = cleaner.standardize_missing(df_raw.copy())

print('Missing values before standardization:', df_raw.isnull().sum().sum())
print('Missing values after  standardization:', df_standardized.isnull().sum().sum())

---
## 3. Remove Duplicate Records


In [ ]:
rows_before = len(df_standardized)
df_deduped = cleaner.remove_duplicates(df_standardized)
rows_after = len(df_deduped)

print(f'Rows removed : {rows_before - rows_after}')
print(f'Rows remaining: {rows_after:,}')

---
## 4. KNN Imputation (k = 5)

KNN imputation estimates missing values by averaging the `k` nearest
neighbors (based on the other non-missing features). This is more
sophisticated than mean/median imputation because it considers
the local structure of the data.


In [ ]:
target_col = 'Target'

# Separate features and target before imputation
# (we never impute the label column itself)
y_series = df_deduped[target_col].copy()
X_to_impute = df_deduped.drop(columns=[target_col])

# Identify numeric columns — only these are imputed with KNN
numeric_cols = X_to_impute.select_dtypes(include=[np.number]).columns.tolist()
print(f'Columns to KNN-impute ({len(numeric_cols)}): {numeric_cols}')

missing_before = X_to_impute[numeric_cols].isnull().sum().sum()
print(f'Missing values before KNN imputation: {missing_before:,}')

In [ ]:
# Apply KNN imputation to the numeric feature columns
X_imputed = cleaner.impute_knn(X_to_impute, exclude_cols=[])

missing_after = X_imputed[numeric_cols].isnull().sum().sum()
print(f'Missing values after KNN imputation : {missing_after:,}')

# Recombine with the target column
df_imputed = pd.concat([X_imputed, y_series.reset_index(drop=True)], axis=1)
print(f'Shape after imputation: {df_imputed.shape}')

---
## 5. Encode Categorical Variables


In [ ]:
# Label-encode any remaining object/categorical columns
# (Blood Type, Location, etc.)
cat_cols = df_imputed.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Categorical columns to encode: {cat_cols}')

df_encoded = cleaner.encode_categoricals(df_imputed, exclude_cols=[target_col])
print(f'Shape after encoding: {df_encoded.shape}')
df_encoded.head()

---
## 6. Feature Engineering


In [ ]:
# Add derived RFMTC features:
# - donation_density   = Frequency / Time
# - recency_score      = 1 / (Recency + 1)
# - high_frequency_flag = 1 if Frequency > median(Frequency)

fb = FeatureBuilder(scale_method='standard')
df_featured = fb.add_derived_features(df_encoded)

print(f'Shape after feature engineering: {df_featured.shape}')
print(f'New features added: donation_density, recency_score, high_frequency_flag')
df_featured.head()

---
## 7. Save Processed Dataset


In [ ]:
# Save the fully cleaned and engineered dataset for use in modeling
processed_dir = os.path.join(PROJECT_ROOT, 'data', 'processed')
os.makedirs(processed_dir, exist_ok=True)

output_path = os.path.join(processed_dir, 'blood_donation_clean.csv')
df_featured.to_csv(output_path, index=False)

print(f'Cleaned dataset saved to: {output_path}')
print(f'Final shape: {df_featured.shape}')
df_featured.describe().T.round(3)

---
## 8. Preprocessing Summary

| Step | Action | Result |
|---|---|---|
| 1 | Standardize NaN markers | Uniform `np.nan` throughout |
| 2 | Remove duplicates | Exact duplicate rows removed |
| 3 | KNN Imputation (k=5) | All numeric missing values filled |
| 4 | Label encoding | Blood Type / Location encoded |
| 5 | Feature engineering | 3 derived features added |
| 6 | Save to CSV | `data/processed/blood_donation_clean.csv` |

> Proceed to **Notebook 03** for model training.


---
## 9. Handling Imbalanced Data — SMOTE Oversampling

The target class is imbalanced (more non-donors than donors).
We apply **SMOTE** (Synthetic Minority Over-sampling Technique) to create
synthetic samples for the minority class, balancing the training set
without simply duplicating existing records.

> ⚠️ SMOTE is applied **only on the training set** — never on the test set.
> Applying it to test data would cause data leakage.


In [ ]:
# ── SMOTE Oversampling on Training Data ──────────────────────────────────────
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

# Re-load the processed dataset
processed_path = os.path.join(PROJECT_ROOT, 'data', 'processed', 'blood_donation_clean.csv')
df_processed = pd.read_csv(processed_path)

TARGET = 'Target'
X_all = df_processed.drop(columns=[TARGET])
y_all = df_processed[TARGET]

print('Class distribution BEFORE SMOTE:')
print(y_all.value_counts())
print(f'Imbalance ratio: {y_all.value_counts()[0] / y_all.value_counts()[1]:.2f}:1')

# Split first — SMOTE only on train
X_tr, X_te, y_tr, y_te = train_test_split(
    X_all, y_all, test_size=0.20, random_state=42, stratify=y_all
)

# Apply SMOTE to training set only
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_tr, y_tr)

print(f'\nTraining set BEFORE SMOTE: {X_tr.shape[0]} samples')
print(f'Training set AFTER  SMOTE: {X_train_sm.shape[0]} samples')
print('\nClass distribution AFTER SMOTE (training only):')
print(pd.Series(y_train_sm).value_counts())


In [ ]:
# ── Visualize class balance before vs after SMOTE ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.patch.set_facecolor('#1E293B')

before_counts = y_tr.value_counts().sort_index()
after_counts  = pd.Series(y_train_sm).value_counts().sort_index()

for ax, counts, title in [
    (axes[0], before_counts, 'Before SMOTE (Training)'),
    (axes[1], after_counts,  'After SMOTE (Training)'),
]:
    ax.set_facecolor('#1E293B')
    bars = ax.bar(["Won't Donate", 'Will Donate'], counts.values,
                  color=['#E74C3C', '#2ECC71'], edgecolor='#0F172A', width=0.5)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + counts.sum()*0.01,
                f'{val:,}', ha='center', va='bottom',
                fontsize=12, fontweight='bold', color='white')
    ax.set_title(title, color='white', fontsize=12)
    ax.set_ylabel('Count', color='#94A3B8')
    ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_color('#334155')

plt.suptitle('SMOTE: Class Balance Before vs After', color='white', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'reports', 'figures', 'smote_balance.png'),
            dpi=120, bbox_inches='tight', facecolor='#1E293B')
plt.show()
print('Figure saved: reports/figures/smote_balance.png')


---
## 10. Updated Preprocessing Summary

| Step | Action | Result |
|------|--------|--------|
| 1 | Standardize NaN markers | Uniform `np.nan` throughout |
| 2 | Remove duplicates | Exact duplicate rows removed |
| 3 | KNN Imputation (k=5) | All missing values filled |
| 4 | Label Encoding | Categorical columns → numeric |
| 5 | Feature Engineering | 3 new derived features added |
| 6 | StandardScaler | All features scaled |
| 7 | SMOTE | Training set class-balanced |
| 8 | Save processed CSV | `data/processed/blood_donation_clean.csv` |
